In [1]:
import numpy as np
import matplotlib.pyplot as plt
import torch
import sys

sys.path.append("../")
from tqdm import tqdm

DATA_DIR = "../mu3e_trigger_data"
MODEL_DIR = "../models"
PLOTS_DIR = "../plots"
SIGNAL_PIXEL_FILE = f"{DATA_DIR}/sig_only_with_layer_pixel_spacetime.npy"
SIGNAL_MPPC_FILE = f"{DATA_DIR}/sig_only_with_layer_mppc_spacetime.npy"

BACKGROUND_PIXEL_FILE = f"{DATA_DIR}/bg_with_layer_pixel_spacetime.npy"
BACKGROUND_MPPC_FILE = f"{DATA_DIR}/bg_with_layer_mppc_spacetime.npy"


sig_mppc_spacetime = np.load(SIGNAL_MPPC_FILE)
sig_pixel_spacetime = np.load(SIGNAL_PIXEL_FILE)
bg_pixel_spacetime = np.load(BACKGROUND_PIXEL_FILE)
bg_mppc_spacetime = np.load(BACKGROUND_MPPC_FILE)

X_pixel = np.concatenate([sig_pixel_spacetime, bg_pixel_spacetime], axis=0)
X_mppc = np.concatenate([sig_mppc_spacetime, bg_mppc_spacetime], axis=0)
y = np.concatenate(
    [np.ones(sig_pixel_spacetime.shape[0]), np.zeros(bg_pixel_spacetime.shape[0])],
    axis=0,
)

from sklearn.model_selection import train_test_split

X_pixel_train, X_pixel_test, X_mppc_train, X_mppc_test, y_train, y_test = train_test_split(
    X_pixel, X_mppc, y, test_size=0.2, random_state=42, stratify=y
)


del sig_pixel_spacetime, sig_mppc_spacetime, bg_pixel_spacetime, bg_mppc_spacetime, X_pixel, X_mppc, y

In [2]:
from src.torch.pre_processing import full_event_to_hetero_graphs,batch_full_events_to_hetero_graphs
hetero_graph = batch_full_events_to_hetero_graphs(
    torch.tensor(X_mppc_train[:2], dtype=torch.float32),
    torch.tensor(X_pixel_train[:2], dtype=torch.float32),
    torch.tensor(y_train[:2], dtype=torch.float32),
    connect_layers=True,
    mppc_timing_cutoff=1
)

In [3]:
hetero_graph.metadata

<bound method HeteroData.metadata of HeteroDataBatch(
  event_idx=[3],
  y=[2],
  pixel={
    x=[51, 3],
    batch=[51],
    ptr=[4],
  },
  mppc={
    x=[59, 4],
    batch=[59],
    ptr=[4],
  },
  (pixel, to, pixel)={
    edge_index=[2, 388],
    edge_labels=[388],
  },
  (mppc, to, mppc)={
    edge_index=[2, 682],
    edge_labels=[682],
  },
  (pixel, to, mppc)={
    edge_index=[2, 283],
    edge_labels=[283],
  },
  (mppc, to, pixel)={
    edge_index=[2, 283],
    edge_labels=[283],
  }
)>

In [74]:
from torch_geometric.nn import HeteroLinear, Linear, HeteroDictLinear
from torch_geometric.nn import HeteroConv, GCNConv, SAGEConv, GATConv, Linear, global_mean_pool, global_max_pool
from src.torch.model.components import get_mlp
from torch import nn
from torch_geometric.data import HeteroData

class LearnEdgeAttr(nn.Module):
    def __init__(self, metadata, in_channels_dict, hidden_channels, out_channels):
        """
        Args:
            metadata: graph.metadata() (tuple of (node_types, edge_types))
            in_channels_dict: dict {node_type: feature_dim}
            hidden_channels: hidden dimension for edge MLP
            out_channels: dimension of learned edge attributes
        """
        super().__init__()
        self.edge_mlps = nn.ModuleDict()

        # one MLP per edge type
        for src, rel, dst in metadata[1]:
            in_src = in_channels_dict[src]
            in_dst = in_channels_dict[dst]
            self.edge_mlps[f"{src}__{rel}__{dst}"] = nn.Sequential(
                nn.Linear(in_src + in_dst, hidden_channels),
                nn.ReLU(),
                nn.Linear(hidden_channels, out_channels)
            )

    def forward(self, x_dict, edge_index_dict, data: HeteroData) -> HeteroData:
        """
        Args:
            x_dict: dict {node_type: node_features}
            edge_index_dict: dict {edge_type: edge_index}
            data: HeteroData graph

        Returns:
            data with added edge_attr for each edge_type
        """
        for edge_type, edge_index in edge_index_dict.items():
            src, rel, dst = edge_type
            mlp = self.edge_mlps[f"{src}__{rel}__{dst}"]

            src_x = x_dict[src][edge_index[0]]
            dst_x = x_dict[dst][edge_index[1]]
            edge_feat = mlp(torch.cat([src_x, dst_x], dim=-1))

            # assign edge attributes to the HeteroData object
            data[edge_type].edge_attr = edge_feat

        return data
    

class HeteroGraphClassifier(nn.Module):
    def __init__(self, metadata, in_channels_dict, hidden_channels, out_channels, num_layers=2, aggr_layout : list = None, dropout=0.5):
        """
        Args:
            metadata: graph.metadata() (tuple of (node_types, edge_types))
            in_channels_dict: dict {node_type: feature_dim}
            hidden_channels: hidden dimension for node embeddings
            out_channels: number of output classes
            num_layers: number of GNN layers
            dropout: dropout rate
        """
        super().__init__()
        if aggr_layout is None:
            aggr_layout = ["mean"] * num_layers
        if len(aggr_layout) != num_layers:
            raise ValueError("Length of aggr_layout must be equal to num_layers")
        


        self.learn_edge_attr = LearnEdgeAttr(metadata, in_channels_dict , hidden_channels, hidden_channels)
        self.lin_dict = nn.ModuleDict()
        for node_type in metadata[0]:
            self.lin_dict[node_type] = nn.Linear(in_channels_dict[node_type], hidden_channels)
      
        self.convs = nn.ModuleList()
        self.linear_residuals = nn.ModuleList()

        for _ in range(num_layers):
            in_channels_dict = {node_type: hidden_channels for node_type in metadata[0]}
            conv = HeteroConv({
                edge_type: GATConv((-1, -1), hidden_channels, heads=4, dropout=dropout, add_self_loops=False)
                for edge_type in metadata[1]
            }, aggr='mean')
            self.convs.append(conv)
            self.linear_residuals.append(HeteroDictLinear({node_type : -1 for node_type in metadata[0]}, hidden_channels))

        self.classifier = nn.Sequential(
            nn.Linear(2 * hidden_channels * len(metadata[0]), hidden_channels),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_channels, out_channels)
        )

        self.dropout = dropout

    def forward(self, data: HeteroData) -> torch.Tensor:
        """
        Args:
            data: HeteroData graph

        Returns:
            logits for each graph in the batch
        """
        x_dict, edge_index_dict = data.x_dict, data.edge_index_dict


        data = self.learn_edge_attr(x_dict, edge_index_dict, data)
        edge_attr_dict = {edge_type: data[edge_type].edge_attr for edge_type in edge_index_dict.keys()}
        print("Edge attributes learned")  # --- DEBUG ---

        # Initial linear transformation for each node type
        x_dict = {
            node_type: self.lin_dict[node_type](x)
            for node_type, x in x_dict.items()
        }
        print("Linear transformations done")  # --- DEBUG ---

        # GNN layers

        residue = x_dict  # Initial residue is the input features
        for conv, linear_residual in zip(self.convs, self.linear_residuals):
            x_dict = conv(x_dict, edge_index_dict, edge_attr_dict=edge_attr_dict)
            # Activation and dropout
            x_dict = {key: x.relu() for key, x in x_dict.items()}
            x_dict = {key: nn.functional.dropout(x, p=self.dropout, training=self.training) for key, x in x_dict.items()}
            # Residual connection with linear transformation
            x_dict = linear_residual(x_dict)
            x_dict = {key: x + residue[key] for key, x in x_dict.items()}
        print("GNN layers done")  # --- DEBUG ---

        # Global mean pooling for each node type
        mean_poolings = []
        max_poolings = []
        for node_type, x in x_dict.items():
            batch = data[node_type].batch
            mean_pool = global_mean_pool(x, batch)
            max_pool = global_max_pool(x, batch)
            mean_poolings.append(mean_pool)
            max_poolings.append(max_pool)
        graph_representation = torch.cat(mean_poolings + max_poolings, dim=-1)
        logits = self.classifier(graph_representation)
        return logits

    

In [75]:
classifier = HeteroGraphClassifier(
    metadata=hetero_graph.metadata(),
    in_channels_dict={
        "mppc": hetero_graph["mppc"].x.shape[1],
        "pixel": hetero_graph["pixel"].x.shape[1]
    },
    hidden_channels=64,
    out_channels=1,
    num_layers=3,
    aggr_layout=["mean", "mean", "mean"],
    dropout=0.5
)